In [1]:
from youtube_comment_downloader import YoutubeCommentDownloader
import pandas as pd
from tqdm import tqdm


def get_youtube_comments_to_df(url, max_comments=1000):
    downloader = YoutubeCommentDownloader()
    comments = downloader.get_comments_from_url(url)
    data = []
    for i, comment in zip(tqdm(range(max_comments),
    desc="Mengambil komentar"), comments):
        data.append({
    'cid': comment.get('cid', ''),
    'username': comment.get('author', ''),
    'comment': comment.get('text', ''),
    'time': comment.get('time', ''),
    'channel': comment.get('channel', ''),
    'votes': comment.get('votes', '0'),
    'replies': comment.get('replies', ''),
    'photo': comment.get('photo', ''),
    'heart': comment.get('heart', False),
    'reply': comment.get('reply', False),
    'time_parsed': comment.get('time_parsed', 0)
    })
    df = pd.DataFrame(data)
    return df


if __name__ == "__main__":
    url = input("Masukkan URL video YouTube: https://youtu.be/ueeaNN9dAPc?si=w1igjbT7Mj_9sr41 ")
    max_comments = int(input("Masukkan jumlah komentar yang ingin diambil (default 1000): ") or 1000)
    df = get_youtube_comments_to_df(url, max_comments)
    print(df)

    df.to_csv("youtube_comments.csv", index=False)

KeyboardInterrupt: 

In [ ]:
from instagrapi import Client
import pandas as pd

def get_ig_comments_instagrapi(shortcode, max_comments=100):
    cl = Client()
    
    # Masukkan Session ID yang sama dari browser
    SESSION_ID = "26843997546%3A1XgSw0arK0d4Dn%3A29%3AAYmRu8OkP09-KTeRZxG-aUOsrT6Ifn9kFw7S_Gow0g"
    
    print("Menghubungkan ke Instagram menggunakan Session ID...")
    try:
        cl.login_by_sessionid(SESSION_ID)
    except Exception as e:
        print(f"Gagal terhubung dengan Session ID: {e}")
        return pd.DataFrame()

    print(f"Mencari data untuk post: {shortcode}...")
    try:
        # Konversi shortcode (URL) menjadi ID internal Instagram
        media_id = cl.media_pk_from_code(shortcode)
    except Exception as e:
        print(f"Gagal menemukan post (mungkin privat atau salah URL): {e}")
        return pd.DataFrame()

    print("Mengambil komentar (proses ini mungkin memakan waktu)...")
    try:
        # Mengambil komentar langsung menggunakan API Mobile
        comments = cl.media_comments(media_id, amount=max_comments)
    except Exception as e:
        print(f"Gagal mengambil komentar: {e}")
        return pd.DataFrame()

    data = []
    for comment in comments:
        data.append({
            'username': comment.user.username,
            'comment': comment.text,
            'time': comment.created_at_utc
        })

    df = pd.DataFrame(data)
    return df

if __name__ == "__main__":
    default_shortcode = "DcIxjrchE9c"
    shortcode = input(f"Masukkan shortcode (Enter untuk {default_shortcode}): ") or default_shortcode
    max_comments = int(input("Masukkan batas komentar (default 100): ") or 100)
    
    df = get_ig_comments_instagrapi(shortcode, max_comments)
    
    if not df.empty:
        print(df.head())
        
        # Hapus informasi zona waktu agar bisa disimpan ke Excel
        df['time'] = df['time'].dt.tz_localize(None)
        
        # Simpan hasil
        df.to_csv("Purple_Cloud_Series_Kopi_kenagan.csv", index=False)
        #df.to_excel("ig_comments.xlsx", index=False)
        print(f"Berhasil menyimpan {len(df)} komentar ke CSV dan Excel.")
    else:
        print("Gagal mengambil data. DataFrame kosong.")

Menghubungkan ke Instagram menggunakan Session ID...
Mencari data untuk post: Ddice13GmJF...
Mengambil komentar (proses ini mungkin memakan waktu)...
      username                                            comment  \
0     wendakio      Siapa kmrn yg kebakaran perkara lope ungu wkw   
1  imyourllve_  buat kakak² yg gatau ini namanya ganti rasa ya...   
2    xyzyaa005  Ada gak sih jangan menu  Mulu langsung bawa ha...   
3    chi.jm_13              Ada gift nya ga min kayak sebelumnya?   
4    tiaraanjn                         ada merch lagi gaa min ???   

                       time  
0 2026-09-21 06:17:16+00:00  
1 2026-09-21 07:32:25+00:00  
2 2026-09-21 06:01:39+00:00  
3 2026-09-21 08:40:23+00:00  
4 2026-09-21 05:53:38+00:00  
Berhasil menyimpan 980 komentar ke CSV dan Excel.


In [ ]:

import os
import random
import re
import time
from getpass import getpass

import pandas as pd
from instagrapi import Client

DEFAULT_SHORTCODE = "Dd-jC5gMUrO"
OUTPUT_CSV = "Kopi_kenangan_comentar_tiga.csv"
OUTPUT_XLSX = "Kopi_kenangan_comentar_tiga.xlsx"

# Karakter kontrol yang bikin openpyxl error saat menulis Excel
ILLEGAL_CHARS = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")


# --------------------------------------------------------------------------- #
# Login
# --------------------------------------------------------------------------- #
def login() -> Client:
    cl = Client()
    cl.delay_range = [2, 5]  # jeda acak antar request internal instagrapi

    session_id = os.environ.get("IG_SESSIONID") or getpass(
        "Masukkan Session ID (input tidak ditampilkan): "
    )
    session_id = session_id.strip()
    if not session_id:
        raise SystemExit("Session ID kosong.")

    print("Menghubungkan ke Instagram menggunakan Session ID...")
    cl.login_by_sessionid(session_id)
    return cl


# --------------------------------------------------------------------------- #
# Helper
# --------------------------------------------------------------------------- #
def clean_text(text) -> str:
    return ILLEGAL_CHARS.sub("", text or "")


def fetch_replies(cl: Client, media_id: str, comment, max_replies: int, retries: int = 2):
    """Ambil balasan satu komentar. Error ditampilkan, tidak disembunyikan."""
    for attempt in range(retries + 1):
        try:
            return cl.media_comment_replies(media_id, str(comment.pk), amount=max_replies)
        except Exception as e:
            wait = 30 * (attempt + 1)
            print(
                f"  [!] Gagal ambil balasan komentar {comment.pk} "
                f"(percobaan {attempt + 1}/{retries + 1}): {type(e).__name__}: {e}"
            )
            if attempt < retries:
                print(f"      Menunggu {wait} detik sebelum mencoba lagi...")
                time.sleep(wait)
    return []


def row(comment, parent=None) -> dict:
    return {
        "comment_id": str(comment.pk),
        "parent_id": str(parent.pk) if parent is not None else None,
        "type": f"Reply to @{parent.user.username}" if parent is not None else "Main Comment",
        "username": comment.user.username,
        "comment": clean_text(comment.text),
        "like_count": getattr(comment, "like_count", None),
        "time": comment.created_at_utc,
    }


def save(data: list):
    df = pd.DataFrame(data)
    if df.empty:
        print("Tidak ada data untuk disimpan.")
        return df
    try:
        df["time"] = pd.to_datetime(df["time"]).dt.tz_localize(None)
    except (TypeError, AttributeError):
        pass

    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")  # utf-8-sig: emoji/aksara aman di Excel
    df.to_excel(OUTPUT_XLSX, index=False)
    print(f"Berhasil menyimpan {len(df)} baris ke {OUTPUT_CSV} dan {OUTPUT_XLSX}.")
    return df


# --------------------------------------------------------------------------- #
# Proses utama
# --------------------------------------------------------------------------- #
def get_ig_deep_comments(shortcode: str, max_comments: int = 100, max_replies: int = 0):
    cl = login()

    if not hasattr(cl, "media_comment_replies"):
        raise SystemExit(
            "Versi instagrapi Anda belum punya media_comment_replies.\n"
            "Jalankan: pip install -U instagrapi"
        )

    print(f"Mencari data untuk post: {shortcode}...")
    media_id = str(cl.media_pk_from_code(shortcode))

    print(f"Mengambil maksimal {max_comments} komentar utama...")
    comments = cl.media_comments(media_id, amount=max_comments)
    print(f"Dapat {len(comments)} komentar utama. Menelusuri balasannya...")

    data = []
    total_replies = 0

    try:
        for i, comment in enumerate(comments, start=1):
            data.append(row(comment))

            # Lewati komentar yang jelas tidak punya balasan (kalau atributnya ada)
            child_count = getattr(comment, "child_comment_count", None)
            if child_count == 0:
                continue

            time.sleep(random.uniform(2, 5))  # jeda acak supaya tidak kena rate limit
            replies = fetch_replies(cl, media_id, comment, max_replies)

            for reply in replies:
                data.append(row(reply, parent=comment))
            total_replies += len(replies)

            if replies:
                print(
                    f"[{i}/{len(comments)}] @{comment.user.username}: "
                    f"{len(replies)} balasan"
                )

    except KeyboardInterrupt:
        print("\nDihentikan manual. Menyimpan data yang sudah terkumpul...")

    print(f"Selesai: {len(comments)} komentar utama + {total_replies} balasan.")
    return save(data)


if __name__ == "__main__":
    shortcode = input(f"Masukkan shortcode (Enter untuk {DEFAULT_SHORTCODE}): ").strip() or DEFAULT_SHORTCODE
    max_comments = int(input("Batas komentar utama (default 100): ").strip() or 100)
    max_replies = int(
        input("Batas balasan per komentar (0 = semua, default 0): ").strip() or 0
    )

    df = get_ig_deep_comments(shortcode, max_comments, max_replies)
    if not df.empty:
        print(df.head())

Menghubungkan ke Instagram menggunakan Session ID...
Mencari data untuk post: DdAq5FVGg1Y...
Mengambil maksimal 200 komentar utama...
Dapat 200 komentar utama. Menelusuri balasannya...
[1/200] @craftaliaa: 2 balasan
[2/200] @hfzsn: 1 balasan
[3/200] @oppitta_: 4 balasan
[4/200] @onyourdes_ss: 3 balasan
[5/200] @seccraaaw__: 6 balasan
[7/200] @nanale.jpg: 1 balasan
[8/200] @hersyaaa_12: 16 balasan
[9/200] @haea.ul: 2 balasan
[10/200] @cyutinis: 13 balasan
[11/200] @suciartkdwn: 6 balasan
[12/200] @siyeph1992: 2 balasan
[13/200] @sti.nurhikmah: 3 balasan
[14/200] @meamengme: 3 balasan
[15/200] @yultagram: 3 balasan
[16/200] @haechildrafts: 1 balasan
[17/200] @teolaegi_bbh0605: 2 balasan
[19/200] @suciimayrani: 2 balasan
[20/200] @pratiwiabc_: 1 balasan
[22/200] @nis_nissaaaaar: 7 balasan
[23/200] @shymeu_: 5 balasan
[25/200] @anissa.zao: 24 balasan
[27/200] @itsmekhayra_: 2 balasan
[28/200] @biia.nn02: 2 balasan
[29/200] @taeraqim: 2 balasan
[30/200] @nafaalfitri2: 7 balasan
[32/200] @qu

In [ ]:
from youtube_comment_downloader import YoutubeCommentDownloader
import pandas as pd
from tqdm import tqdm

def get_youtube_comments_to_df(url, max_comments=1000):
    downloader = YoutubeCommentDownloader()
    comments = downloader.get_comments_from_url(url)
    data = []
    
    # Mengambil komentar sesuai batas max_comments
    for i, comment in zip(tqdm(range(max_comments), desc="Mengambil komentar"), comments):
        data.append({
            'cid': comment.get('cid', ''),
            'username': comment.get('author', ''),
            'comment': comment.get('text', ''),
            'time': comment.get('time', ''),
            'channel': comment.get('channel', ''),
            'votes': comment.get('votes', '0'),
            'replies': comment.get('replies', ''),
            'photo': comment.get('photo', ''),
            'heart': comment.get('heart', False),
            'reply': comment.get('reply', False),
            'time_parsed': comment.get('time_parsed', 0)
        })
        
    # Perbaikan: 'return df' harus sejajar dengan perulangan for (di luar loop)
    df = pd.DataFrame(data)
    return df

if __name__ == "__main__":
    url = input("Masukkan URL video YouTube: ")
    max_c = input("Masukkan jumlah komentar yang ingin diambil (default 1000): ")
    limit = int(max_c) if max_c.strip() else 1000
    
    df = get_youtube_comments_to_df(url, limit)
    
    print("\nPratinjau Hasil:")
    print(df.head())
    
    # Simpan ke CSV
    df.to_csv("yt_comments.csv", index=False)
    print(f"\nBerhasil mengambil {len(df)} komentar dan menyimpannya ke 'yt_comments.csv'!")

Mengambil komentar: 100%|██████████| 100/100 [00:15<00:00,  6.54it/s]



Pratinjau Hasil:
                          cid               username  \
0  UgzzoW7UJqPHbn4hmN54AaABAg         @LonnieAguirre   
1  Ugyr9tqsEUVsB_Wj3Zl4AaABAg        @qiiiiiiiii-x7s   
2  UgzcSVTo4kK8cXFCTNJ4AaABAg               @neotiar   
3  Ugz-Ioir_RSMhju_E794AaABAg  @FarhanRamadhan-ft6pq   
4  Ugzg7WJ2o0bM6JBUV6R4AaABAg  @FarhanRamadhan-ft6pq   

                                             comment              time  \
0  This tutorial is clear and easy to follow I wi...   9 jam yang lalu   
1                       ringkasan yang gak ringkas 🥀   9 jam yang lalu   
2                              Udah 2x baca buku ini  1 hari yang lalu   
3                              Saya nonton ini 2026😅  3 hari yang lalu   
4                                                 😮😅  3 hari yang lalu   

                    channel votes replies  \
0  UC63T8SDKCFmscCjfjQHQc6A     1       1   
1  UCf3awWTzZo3qUAhkT1BIsxQ     0           
2  UCuDKd_3YQpId56UGp8HbkHg     1       1   
3  UCddFiwDeYOfIPxFe

In [ ]:
"""
Pengambil data X (Twitter) dengan login sesi (cookie akun sendiri), tanpa API resmi.
Hanya mengumpulkan data, tanpa analisis. Menggunakan library tidak resmi `twikit`.

Instalasi (PENTING: pakai fork `twifork`, karena twikit resmi rusak sejak Maret 2026):
    pip uninstall -y twikit
    pip install -U "twifork[impersonate]" pandas openpyxl
    (di notebook pakai %pip, lalu Restart Kernel)

Cara mengambil cookie (mirip Session ID di Instagram):
    1. Buka x.com dan login.
    2. Tekan F12 -> tab Application -> Cookies -> https://x.com
    3. Salin nilai `auth_token` dan `ct0`.
    4. Isi saat diminta, atau set environment variable X_AUTH_TOKEN dan X_CT0.

PERINGATAN:
    - Cookie = kunci masuk akun Anda. Jangan dibagikan dan jangan di-commit ke Git.
    - Akses seperti ini melanggar ketentuan X. Akun bisa dibatasi atau ditangguhkan.
      Sebaiknya pakai akun cadangan, dan jangan menaikkan kecepatan (jeda sudah diacak).
    - Library ini meniru API internal X, jadi bisa rusak sewaktu-waktu saat X berubah.
      Kalau error aneh muncul: pip install -U twifork.

Dua mode:
    1) Cari tweet berdasarkan kata kunci (bisa dibatasi rentang tanggal, tanpa batas 7 hari)
    2) Ambil balasan dari satu tweet (URL atau ID tweet)
"""

import asyncio
import os
import random
import re
import threading
import time
from datetime import datetime
from getpass import getpass

import pandas as pd
from twikit import Client
from twikit.errors import AccountLocked, AccountSuspended, TooManyRequests, Unauthorized

OUTPUT_CSV = "X_data_sesi.csv"
OUTPUT_XLSX = "X_data_sesi.xlsx"

# Karakter kontrol yang bikin openpyxl error saat menulis Excel
ILLEGAL_CHARS = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")


# --------------------------------------------------------------------------- #
# Input
# --------------------------------------------------------------------------- #
def ask_int(prompt: str, default: int) -> int:
    return int(input(prompt).strip() or default)


def ask_date(prompt: str) -> str:
    """Minta tanggal YYYY-MM-DD. Kosong = tidak dibatasi."""
    value = input(prompt).strip()
    if not value:
        return ""
    try:
        datetime.strptime(value, "%Y-%m-%d")
    except ValueError:
        raise SystemExit("Format tanggal harus YYYY-MM-DD, contoh: 2025-01-31.")
    return value


def parse_tweet_id(value: str) -> str:
    """Terima URL tweet atau ID angka, kembalikan ID sebagai string."""
    match = re.search(r"status/(\d+)", value) or re.fullmatch(r"\s*(\d+)\s*", value)
    if not match:
        raise SystemExit("URL/ID tweet tidak valid.")
    return match.group(1)


def clean_cookie(value: str) -> str:
    """Bersihkan nilai cookie dari spasi, kutip, dan awalan 'nama=' hasil salin-tempel."""
    value = value.strip().strip("\"'")
    if value.lower().startswith(("auth_token=", "ct0=")):
        value = value.split("=", 1)[1]
    return value.strip().strip(";").strip("\"'")


def get_cookies() -> tuple:
    auth_token = clean_cookie(
        os.environ.get("X_AUTH_TOKEN")
        or getpass("Masukkan cookie auth_token (input tidak ditampilkan): ")
    )
    ct0 = clean_cookie(
        os.environ.get("X_CT0") or getpass("Masukkan cookie ct0 (input tidak ditampilkan): ")
    )
    if not auth_token or not ct0:
        raise SystemExit("auth_token / ct0 kosong.")
    # Hanya panjangnya yang ditampilkan, nilainya tidak.
    print(
        f"Panjang cookie: auth_token={len(auth_token)} (biasanya 40), "
        f"ct0={len(ct0)} (biasanya 160)"
    )
    return auth_token, ct0


# --------------------------------------------------------------------------- #
# Helper
# --------------------------------------------------------------------------- #
def clean_text(text) -> str:
    return ILLEGAL_CHARS.sub("", text or "")


def to_row(tweet, parent_id=None, type_="Tweet") -> dict:
    return {
        "comment_id": str(tweet.id),
        "parent_id": parent_id,
        "type": type_,
        "username": tweet.user.screen_name if tweet.user else None,
        "comment": clean_text(tweet.full_text or tweet.text),
        "like_count": tweet.favorite_count,
        "retweet_count": tweet.retweet_count,
        "reply_count": tweet.reply_count,
        "quote_count": tweet.quote_count,
        "view_count": tweet.view_count,
        "time": tweet.created_at_datetime,
    }


async def with_retry(factory, what: str, retries: int = 2):
    """Jalankan request dengan retry. Error ditampilkan, tidak disembunyikan."""
    for attempt in range(retries + 1):
        try:
            return await factory()
        except TooManyRequests as e:
            reset = getattr(e, "rate_limit_reset", None)
            wait = 900
            if isinstance(reset, (int, float)) and reset > time.time():
                wait = min(int(reset - time.time()) + 5, 1800)
            print(f"  [!] Kena rate limit saat {what}. Menunggu {wait} detik...")
            await asyncio.sleep(wait)
        except (Unauthorized, AccountLocked, AccountSuspended):
            raise
        except Exception as e:
            wait = 30 * (attempt + 1)
            print(
                f"  [!] Gagal {what} (percobaan {attempt + 1}/{retries + 1}): "
                f"{type(e).__name__}: {e}"
            )
            if attempt < retries:
                print(f"      Menunggu {wait} detik sebelum mencoba lagi...")
                await asyncio.sleep(wait)
    return None


async def iter_tweets(page, limit: int, what: str):
    """Telusuri halaman demi halaman (Result.next) sampai `limit` tweet."""
    count = 0
    while page and count < limit:
        for tweet in page:
            yield tweet
            count += 1
            if count >= limit:
                return
        print(f"  terkumpul {count} tweet...")
        await asyncio.sleep(random.uniform(2, 5))  # jeda acak supaya tidak kena rate limit
        page = await with_retry(page.next, what)


def make_client() -> Client:
    """Buat client. Pakai browser-TLS impersonation kalau tersedia (twifork[impersonate])."""
    try:
        client = Client("en-US", impersonate="chrome124")
        print("Impersonation browser: AKTIF")
        return client
    except (TypeError, ImportError) as e:
        print(
            f"Impersonation browser: TIDAK AKTIF ({type(e).__name__}). "
            "Pasang dengan: pip install -U curl_cffi"
        )
        return Client("en-US")


async def login(auth_token: str, ct0: str) -> Client:
    client = make_client()
    client.set_cookies({"auth_token": auth_token, "ct0": ct0})

    print("Menghubungkan ke X menggunakan cookie sesi...")
    try:
        if hasattr(client, "is_logged_in"):  # tersedia di twifork
            # Cek ini memakai endpoint v1.1 yang bisa ditolak Cloudflare walau cookie
            # benar, jadi hasil False hanya dijadikan peringatan, bukan penghenti.
            if await client.is_logged_in():
                print("Cookie valid.")
            else:
                print(
                    "  [!] Pengecekan login gagal. Penyebabnya bisa cookie salah/kedaluwarsa, "
                    "atau impersonation tidak aktif. Tetap dicoba..."
                )
        else:
            me = await client.user()
            print(f"Terhubung sebagai @{me.screen_name}")
    except (Unauthorized, AccountLocked, AccountSuspended) as e:
        raise SystemExit(f"Cookie tidak valid / kedaluwarsa / akun terkunci: {type(e).__name__}: {e}")
    except Exception as e:
        print(f"  [!] Tidak bisa memverifikasi login ({type(e).__name__}: {e}), tetap dilanjutkan...")
    return client


# --------------------------------------------------------------------------- #
# Mode 1: cari berdasarkan kata kunci
# --------------------------------------------------------------------------- #
async def scrape_search(client, keyword, lang, since, until, max_items, data):
    query = f"({keyword})"
    if lang:
        query += f" lang:{lang}"
    if since:
        query += f" since:{since}"
    if until:
        query += f" until:{until}"
    query += " -filter:retweets"

    print(f"Mencari tweet: {query}")
    page = await with_retry(lambda: client.search_tweet(query, "Latest", count=20), "mencari tweet")

    seen = set()
    async for tweet in iter_tweets(page, max_items, "mengambil halaman berikutnya"):
        if tweet.id in seen:
            continue
        seen.add(tweet.id)
        data.append(to_row(tweet))

    print(f"Selesai: {len(data)} tweet.")


# --------------------------------------------------------------------------- #
# Mode 2: balasan dari satu tweet
# --------------------------------------------------------------------------- #
async def scrape_replies(client, root_id, max_comments, max_nested, data):
    print(f"Mengambil tweet {root_id}...")
    root = await with_retry(lambda: client.get_tweet_by_id(root_id), "mengambil tweet asal")
    if root is None:
        print("Tweet asal tidak bisa diambil.")
        return

    print(f"Tweet dari @{root.user.screen_name}, jumlah balasan menurut X: {root.reply_count}")
    print(f"Mengambil maksimal {max_comments} komentar utama...")

    seen = {str(root.id)}
    id_to_user = {}
    comments = []

    async for tweet in iter_tweets(root.replies, max_comments, "mengambil balasan berikutnya"):
        if str(tweet.id) in seen:
            continue
        seen.add(str(tweet.id))

        username = tweet.user.screen_name if tweet.user else None
        id_to_user[str(tweet.id)] = username

        parent = str(tweet.in_reply_to) if tweet.in_reply_to else None
        if parent is None or parent == str(root.id):
            data.append(to_row(tweet, type_="Main Comment"))
            comments.append(tweet)
        else:
            parent_user = id_to_user.get(parent)
            type_ = f"Reply to @{parent_user}" if parent_user else "Reply"
            data.append(to_row(tweet, parent_id=parent, type_=type_))

    print(f"Dapat {len(comments)} komentar utama.")

    if max_nested <= 0:
        return

    print("Menelusuri balasan bertingkat...")
    total_nested = 0
    for i, comment in enumerate(comments, start=1):
        if not comment.reply_count:
            continue

        await asyncio.sleep(random.uniform(2, 5))
        thread = await with_retry(
            lambda: client.get_tweet_by_id(comment.id), f"ambil balasan komentar {comment.id}"
        )
        if thread is None or not thread.replies:
            continue

        found = 0
        async for reply in iter_tweets(thread.replies, max_nested, "mengambil balasan bertingkat"):
            if str(reply.id) in seen:
                continue
            seen.add(str(reply.id))
            data.append(
                to_row(
                    reply,
                    parent_id=str(comment.id),
                    type_=f"Reply to @{comment.user.screen_name}",
                )
            )
            found += 1

        total_nested += found
        if found:
            print(f"[{i}/{len(comments)}] @{comment.user.screen_name}: {found} balasan")

    print(f"Selesai: {len(comments)} komentar utama + {total_nested} balasan bertingkat.")


# --------------------------------------------------------------------------- #
# Simpan
# --------------------------------------------------------------------------- #
def save(data: list):
    df = pd.DataFrame(data)
    if df.empty:
        print("Tidak ada data untuk disimpan.")
        return df

    df = df.drop_duplicates(subset="comment_id").reset_index(drop=True)
    try:
        df["time"] = pd.to_datetime(df["time"]).dt.tz_localize(None)
    except (TypeError, AttributeError):
        pass

    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")  # utf-8-sig: emoji aman di Excel
    df.to_excel(OUTPUT_XLSX, index=False)
    print(f"Berhasil menyimpan {len(df)} baris ke {OUTPUT_CSV} dan {OUTPUT_XLSX}.")
    return df


# --------------------------------------------------------------------------- #
# Proses utama
# --------------------------------------------------------------------------- #
def run_sync(coro):
    """Jalankan coroutine. Aman dipakai di script biasa maupun Jupyter/VS Code notebook."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)  # script biasa (terminal)

    # Notebook: sudah ada event loop aktif -> jalankan di thread terpisah
    box = {}
    loop = asyncio.new_event_loop()

    def worker():
        asyncio.set_event_loop(loop)
        try:
            box["result"] = loop.run_until_complete(coro)
        except BaseException as e:
            box["error"] = e
        finally:
            loop.close()

    thread = threading.Thread(target=worker, daemon=True)
    thread.start()
    try:
        while thread.is_alive():
            thread.join(timeout=0.5)
    except KeyboardInterrupt:
        # Tombol Interrupt di notebook: hentikan semua task di thread
        def cancel_all():
            for task in asyncio.all_tasks(loop):
                task.cancel()

        loop.call_soon_threadsafe(cancel_all)
        thread.join(timeout=10)
        raise

    if "error" in box:
        raise box["error"]
    return box.get("result")


async def run(mode, params, cookies, data):
    client = await login(*cookies)
    try:
        if mode == "2":
            await scrape_replies(
                client, params["root_id"], params["max_comments"], params["max_nested"], data
            )
        else:
            await scrape_search(
                client,
                params["keyword"],
                params["lang"],
                params["since"],
                params["until"],
                params["max_items"],
                data,
            )
    except (Unauthorized, AccountLocked, AccountSuspended) as e:
        print(
            f"\n[!] X menolak sesi ini ({type(e).__name__}). Kemungkinan cookie salah atau "
            "kedaluwarsa, atau akun terkunci. Ambil ulang auth_token dan ct0 dari browser."
        )


if __name__ == "__main__":
    print("Mode:\n  1) Cari tweet berdasarkan kata kunci\n  2) Ambil balasan dari satu tweet")
    mode = input("Pilih mode (1/2, default 1): ").strip() or "1"

    if mode == "2":
        params = {
            "root_id": parse_tweet_id(input("Masukkan URL atau ID tweet: ")),
            "max_comments": ask_int("Batas komentar utama (default 100): ", 100),
            "max_nested": ask_int(
                "Batas balasan bertingkat per komentar (0 = tidak diambil, default 0): ", 0
            ),
        }
    else:
        keyword = input('Masukkan kata kunci (contoh: kopi kenangan OR "kopken"): ').strip()
        if not keyword:
            raise SystemExit("Kata kunci kosong.")
        params = {
            "keyword": keyword,
            "lang": input("Kode bahasa (id/en, Enter = semua bahasa): ").strip(),
            "since": ask_date("Tanggal mulai YYYY-MM-DD (Enter = tanpa batas): "),
            "until": ask_date("Tanggal akhir YYYY-MM-DD (Enter = sekarang): "),
            "max_items": ask_int("Batas jumlah tweet (default 100): ", 100),
        }

    cookies = get_cookies()

    data = []
    try:
        run_sync(run(mode, params, cookies, data))
    except KeyboardInterrupt:
        print("\nDihentikan manual. Menyimpan data yang sudah terkumpul...")

    df = save(data)
    if not df.empty:
        print(df.head())

Mode:
  1) Cari tweet berdasarkan kata kunci
  2) Ambil balasan dari satu tweet
Panjang cookie: auth_token=40 (biasanya 40), ct0=160 (biasanya 160)
Impersonation browser: AKTIF
Menghubungkan ke X menggunakan cookie sesi...
Cookie valid.
Mengambil tweet 2096418574289998266...
Tweet dari @Kopikenangan_ID, jumlah balasan menurut X: 413
Mengambil maksimal 100 komentar utama...
  terkumpul 36 tweet...
  terkumpul 72 tweet...
Dapat 100 komentar utama.


Task was destroyed but it is pending!
task: <Task pending name='Task-37' coro=<AsyncCurl._force_timeout() running at c:\Users\hp\Documents\Pemograman BI\.venv\Lib\site-packages\curl_cffi\aio.py:235> wait_for=<Future pending cb=[Task.task_wakeup()]>>


Berhasil menyimpan 100 baris ke X_data_sesi.csv dan X_data_sesi.xlsx.
            comment_id parent_id          type     username  \
0  2096438459598770447      None  Main Comment   jefdimples   
1  2096430185281462374      None  Main Comment    tyasning3   
2  2096447451511296118      None  Main Comment    sky__6813   
3  2096443203243442285      None  Main Comment   66sunblush   
4  2096427752425758788      None  Main Comment  suhnnybear_   

                                             comment  like_count  \
0  @Kopikenangan_ID Min, tolong cupsleeve nya dip...         261   
1  @Kopikenangan_ID Cup sleeve nya tolong di rest...          50   
2  @Kopikenangan_ID yesss yeesss opsinya gk kudu ...          43   
3  @Kopikenangan_ID gift nya banyak, minumannya b...          11   
4  @Kopikenangan_ID Kita tuntaskan sampai batch t...          10   

   retweet_count  reply_count  quote_count view_count                time  
0              1            0            0       7300 2026-09-06 0

c:\Users\hp\Documents\Pemograman BI\.venv\Scripts\python.exe
c:\Users\hp\Documents\Pemograman BI\.venv\Lib\site-packages\twikit\__init__.py
2.4.0


In [ ]:
"""
Scraper komentar TikTok (versi browser asli).

Beda dengan versi TikTokApi: skrip ini TIDAK memalsukan request ke API TikTok.
Ia membuka video di Chrome asli (profil tersimpan), lalu merekam respons
/api/comment/list/ dan /api/comment/list/reply/ yang dibuat oleh halaman TikTok
sendiri. Tanda tangan request (X-Bogus, msToken, dll.) otomatis valid, jadi
msToken tidak perlu diisi lagi.

Persiapan (sekali saja):
    pip install -U playwright pandas openpyxl
    # Kalau Google Chrome / Edge sudah terpasang, tidak perlu apa-apa lagi.
    # Kalau belum:  python -m playwright install chromium

Catatan penting:
- Jalankan dengan jendela browser terlihat (default). Kalau muncul captcha atau
  pop-up login, selesaikan manual; skrip menunggu. Sesi tersimpan di folder
  PROFILE_DIR, jadi lain kali biasanya tidak diminta lagi.
- Selector tombol "Lihat balasan" dan panel komentar bisa berubah kapan saja
  (TikTok sering ganti tampilan). Semuanya dikumpulkan di bagian KONFIGURASI.
"""
import asyncio
import concurrent.futures
import json
import os
import random
import re
import sys
import time
from datetime import datetime, timezone
from urllib.parse import parse_qs, urlsplit

import pandas as pd
from playwright.async_api import async_playwright

OUTPUT_CSV = "Kopi_kenangan_tiktok_comentar.csv"
OUTPUT_VIDEO_CSV = "Kopi_kenangan_tiktok_video.csv"  # statistik level video (like, share, simpan)
OUTPUT_XLSX = "Kopi_kenangan_tiktok_comentar.xlsx"  # sheet "komentar" + sheet "statistik_video"
DEBUG_SCREENSHOT = "debug_tiktok.png"

# --------------------------------------------------------------------------- #
# KONFIGURASI
# --------------------------------------------------------------------------- #
# Default: browser terlihat (lebih jarang kena captcha & bisa diselesaikan manual).
# Set TIKTOK_HEADLESS=1 kalau mau disembunyikan.
HEADLESS = os.environ.get("TIKTOK_HEADLESS", "0") == "1"

# Folder profil browser (cookie, login, captcha yang sudah diselesaikan)
PROFILE_DIR = os.environ.get("TIKTOK_PROFILE", "tiktok_profile")

# Batas waktu membuka halaman video (ms)
PAGE_TIMEOUT_MS = int(os.environ.get("TIKTOK_TIMEOUT_MS", "120000"))

# Berapa detik menunggu komentar pertama muncul (waktu untuk captcha/login manual)
FIRST_COMMENTS_WAIT_S = int(os.environ.get("TIKTOK_WAIT_S", "180"))

MAX_SCROLL_ROUNDS = 300
MAX_REPLY_ROUNDS = 80
REPLY_CLICKS_PER_ROUND = 8

# Selector komentar di halaman TikTok (bisa berubah)
SEL_COMMENT_ITEM = '[data-e2e="comment-level-1"]'
SEL_CAPTCHA = (
    "#captcha-verify-container, .captcha-verify-container, .captcha_verify_container, "
    'iframe[src*="captcha"], [id^="captcha"]'
)

# Penanda bahwa panel komentar sudah terbuka (dipakai supaya ikon tidak diklik dua kali = menutup panel)
SEL_COMMENT_PANEL_ANY = (
    '[data-e2e="comment-level-1"], [data-e2e="comment-list"], [class*="DivCommentListContainer"]'
)
# Tombol pembuka panel komentar (urut dari yang paling spesifik)
SEL_COMMENT_OPENERS = [
    '[data-e2e="comment-icon"]',
    '[data-e2e="browse-comment-icon"]',
    'button[aria-label*="omentar" i]',
    '[data-e2e="comment-count"]',
]

# Karakter kontrol yang bikin openpyxl error saat menulis Excel
ILLEGAL_CHARS = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

# JS: scroll panel komentar sampai dasar
JS_SCROLL_COMMENTS = """
(sel) => {
  const items = document.querySelectorAll(sel);
  const last = items[items.length - 1];
  if (!last) return false;
  last.scrollIntoView({block: "end"});
  let el = last.parentElement;
  while (el) {
    const oy = getComputedStyle(el).overflowY;
    if (el.scrollHeight > el.clientHeight + 5 && /(auto|scroll)/.test(oy)) {
      el.scrollTop = el.scrollHeight;
      break;
    }
    el = el.parentElement;
  }
  return true;
}
"""

# JS: tandai semua tombol "Lihat N balasan" / "View N replies" (Indonesia & Inggris)
JS_MARK_REPLY_BUTTONS = """
() => {
  document.querySelectorAll('[data-kk-reply]').forEach(e => e.removeAttribute('data-kk-reply'));
  const want = /^(view|lihat)\\b.*(repl|balas)/i;
  const skip = /(hide|sembunyi|tutup)/i;
  const hits = [];
  document.querySelectorAll('div,span,p,button').forEach(el => {
    const t = (el.innerText || '').trim();
    if (!t || t.length > 45 || !want.test(t) || skip.test(t)) return;
    hits.push(el);
  });
  // ambil elemen paling dalam saja (hindari klik dobel pada pembungkusnya)
  const inner = hits.filter(el => !hits.some(o => o !== el && el.contains(o)));
  inner.forEach((el, i) => el.setAttribute('data-kk-reply', String(i)));
  return inner.length;
}
"""


# --------------------------------------------------------------------------- #
# Helper
# --------------------------------------------------------------------------- #
def clean_text(text) -> str:
    return ILLEGAL_CHARS.sub("", text or "")


def to_int(value):
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def parse_video_ref(ref: str) -> dict:
    """Terima URL video (termasuk link pendek vm.tiktok.com) atau ID angka saja."""
    ref = ref.strip()
    if ref.isdigit():
        return {"id": ref, "url": f"https://www.tiktok.com/@_/video/{ref}"}
    m = re.search(r"/video/(\d+)", ref)
    if m:
        return {"id": m.group(1), "url": ref.split("?")[0]}
    return {"id": None, "url": ref}  # link pendek: ID diambil dari URL setelah redirect


def row(d: dict, parent: dict = None) -> dict:
    """Ubah satu objek komentar mentah (dict dari API TikTok) jadi satu baris tabel."""
    user = d.get("user") or {}
    parent_user = (parent.get("user") or {}) if parent is not None else {}

    created = to_int(d.get("create_time"))
    created_utc = datetime.fromtimestamp(created, tz=timezone.utc) if created else None

    return {
        "comment_id": str(d.get("cid", "")),
        "parent_id": str(parent.get("cid", "")) if parent is not None else None,
        "type": f"Reply to @{parent_user.get('unique_id')}" if parent is not None else "Main Comment",
        "username": user.get("unique_id"),
        "comment": clean_text(d.get("text")),
        "like_count": d.get("digg_count"),
        "time": created_utc,
    }


def video_stats_from_item(item: dict) -> dict:
    """Statistik level video dari objek itemStruct: views, like, komentar, share, simpan."""
    if not item:
        return {}
    stats = item.get("stats") or {}
    stats_v2 = item.get("statsV2") or {}

    def pick(key):
        val = to_int(stats_v2.get(key))
        return val if val is not None else to_int(stats.get(key))

    author = item.get("author")
    author = author.get("uniqueId") if isinstance(author, dict) else author

    created = to_int(item.get("createTime"))
    posted_at = (
        datetime.fromtimestamp(created, tz=timezone.utc).replace(tzinfo=None) if created else None
    )

    return {
        "video_id": str(item.get("id") or ""),
        "author": author,
        "description": clean_text(item.get("desc")),
        "posted_at": posted_at,
        "views": pick("playCount"),
        "likes": pick("diggCount"),
        "comments": pick("commentCount"),
        "shares": pick("shareCount"),
        "saves": pick("collectCount"),
    }


def item_from_rehydration(raw_json: str) -> dict:
    """Ambil itemStruct dari <script id="__UNIVERSAL_DATA_FOR_REHYDRATION__"> pada halaman video."""
    if not raw_json:
        return {}
    try:
        data = json.loads(raw_json)
    except ValueError:
        return {}
    scope = data.get("__DEFAULT_SCOPE__") or {}
    detail = scope.get("webapp.video-detail") or {}
    return (detail.get("itemInfo") or {}).get("itemStruct") or {}


# --------------------------------------------------------------------------- #
# Perekam respons API
# --------------------------------------------------------------------------- #
class Collector:
    """Menampung semua respons komentar yang dibuat halaman TikTok."""

    def __init__(self, video_id: str = None):
        self.video_id = video_id
        self.main = {}  # cid -> dict komentar utama (urut kedatangan)
        self.replies = {}  # cid induk -> {cid balasan -> dict}
        self.item = {}  # data video dari /api/item/detail/ (cadangan)
        self.has_more = True
        self.main_events = 0  # berapa kali respons komentar utama diterima
        self.bad_responses = 0  # respons kosong / bukan JSON (tanda diblokir)
        self.seen_urls = []
        self._tasks = set()

    def attach(self, page):
        page.on("response", self._on_response)

    def _on_response(self, response):
        url = response.url
        if "/api/comment/list" not in url and "/api/item/detail" not in url:
            return
        task = asyncio.ensure_future(self._handle(response))
        self._tasks.add(task)
        task.add_done_callback(self._tasks.discard)

    async def _handle(self, response):
        try:
            data = await response.json()
        except Exception:
            self.bad_responses += 1
            return
        self.ingest(response.url, data)

    def ingest(self, url: str, data: dict):
        """Masukkan satu respons API (dipisah dari Playwright supaya mudah diuji)."""
        if not isinstance(data, dict):
            return
        parts = urlsplit(url)
        path = parts.path.rstrip("/")
        query = parse_qs(parts.query)
        if len(self.seen_urls) < 30:
            self.seen_urls.append(path)

        if path.endswith("/api/item/detail"):
            item = (data.get("itemInfo") or {}).get("itemStruct")
            if item:
                self.item = item
            return

        # abaikan komentar milik video lain (misalnya video rekomendasi)
        aweme = (query.get("aweme_id") or [""])[0]
        if self.video_id and aweme and aweme != self.video_id:
            return

        comments = data.get("comments") or []
        if path.endswith("/comment/list/reply"):
            parent = (query.get("comment_id") or [""])[0]
            for c in comments:
                pid = parent or str(c.get("reply_id") or "")
                self.replies.setdefault(pid, {})[str(c.get("cid"))] = c
        elif path.endswith("/comment/list"):
            self.main_events += 1
            self.has_more = bool(data.get("has_more"))
            for c in comments:
                self.main[str(c.get("cid"))] = c

    def reply_count(self) -> int:
        return sum(len(v) for v in self.replies.values())


def build_rows(col: Collector, max_comments: int, max_replies: int):
    """Susun baris: komentar utama diikuti balasannya (urutan sama seperti versi lama)."""
    data = []
    mains = list(col.main.values())[: max_comments or None]
    total_replies = 0
    for c in mains:
        data.append(row(c))
        reps = list(col.replies.get(str(c.get("cid")), {}).values())
        if max_replies:
            reps = reps[:max_replies]
        for r in reps:
            data.append(row(r, parent=c))
        total_replies += len(reps)
    return data, len(mains), total_replies


# --------------------------------------------------------------------------- #
# Browser
# --------------------------------------------------------------------------- #
async def launch_context(p, browser: str, headless: bool):
    """Buka browser asli dengan profil tersimpan. Urutan percobaan: pilihan pengguna, lalu cadangan."""
    order = {"chrome": ["chrome", "msedge", None], "msedge": ["msedge", "chrome", None]}.get(
        browser, [None, "chrome", "msedge"]
    )
    last_err = None
    for channel in order:
        try:
            kwargs = dict(
                user_data_dir=PROFILE_DIR,
                headless=headless,
                no_viewport=True,  # ikuti ukuran jendela asli (viewport tetap = tanda otomatisasi)
                locale="id-ID",
                args=["--disable-blink-features=AutomationControlled", "--start-maximized"],
                ignore_default_args=["--enable-automation"],
            )
            if channel:
                kwargs["channel"] = channel
            ctx = await p.chromium.launch_persistent_context(**kwargs)
            print(f"Browser: {channel or 'chromium bawaan Playwright'}")
            return ctx
        except Exception as e:
            last_err = e
            detail = (str(e).strip().splitlines() or [""])[0]
            print(f"  [!] Gagal membuka {channel or 'chromium'}: {type(e).__name__}: {detail}")
    raise SystemExit(
        "Tidak ada browser yang bisa dibuka. Pasang Google Chrome, atau jalankan: "
        "python -m playwright install chromium. Kalau ada jendela browser dari proses "
        f"sebelumnya yang masih terbuka, tutup dulu. ({type(last_err).__name__})"
    )


async def has_captcha(page) -> bool:
    try:
        return await page.locator(SEL_CAPTCHA).count() > 0
    except Exception:
        return False


async def open_comments_panel(page) -> bool:
    """Klik ikon komentar kalau panel belum terbuka. True kalau ada yang diklik."""
    try:
        if await page.locator(SEL_COMMENT_PANEL_ANY).count() > 0:
            return False  # sudah terbuka (mungkin masih memuat): jangan diklik lagi
    except Exception:
        pass
    for sel in SEL_COMMENT_OPENERS:
        try:
            await page.locator(sel).first.click(timeout=2500)
            return True
        except Exception:
            continue
    return False


async def check_landing(page, col: Collector, ref: dict):
    """Pastikan benar-benar di halaman video. TikTok kadang mengalihkan ke beranda/feed."""

    def video_id_in_url():
        m = re.search(r"/video/(\d+)", page.url)
        return m.group(1) if m else None

    def wrong(vid):
        return vid is None or bool(col.video_id and vid != col.video_id)

    vid = video_id_in_url()
    if wrong(vid):
        print("  [!] TikTok mengalihkan ke halaman lain, bukan halaman video. Mencoba sekali lagi...")
        try:
            await page.goto(ref["url"], wait_until="domcontentloaded", timeout=PAGE_TIMEOUT_MS)
        except Exception:
            pass
        await asyncio.sleep(random.uniform(3, 5))
        vid = video_id_in_url()

    try:
        title = await page.title()
    except Exception:
        title = ""
    print(f"  Halaman: {page.url}")
    print(f"  Judul  : {title}")

    if wrong(vid):
        print("  [!] Halaman yang terbuka BUKAN video target (URL tidak lengkap atau dialihkan ke beranda).")
        print("      Di jendela browser ini, tempel URL lengkap videonya (bentuk .../@username/video/ID)")
        print("      di address bar, lalu klik ikon komentar. Skrip tetap menangkap komentarnya.")
    elif not col.video_id:
        col.video_id = vid  # link pendek: ID diambil dari URL akhir


async def print_dom_hints(page):
    """Bantuan diagnosa: nilai data-e2e yang ada di halaman (untuk menyesuaikan selector)."""
    try:
        names = await page.evaluate(
            "() => [...new Set([...document.querySelectorAll('[data-e2e]')]"
            ".map(e => e.getAttribute('data-e2e')))].slice(0, 80)"
        )
        print(f"  URL akhir: {page.url}")
        print(f"  data-e2e di halaman: {names}")
    except Exception:
        pass


async def wait_for_first_comments(page, col: Collector, headless: bool) -> str:
    """Tunggu respons komentar pertama. Kembalian: "ok", "timeout", atau "captcha_hidden".

    "captcha_hidden" = captcha muncul padahal browser tersembunyi (tidak mungkin diselesaikan),
    pemanggil sebaiknya membuka ulang dengan jendela terlihat.
    """
    start = time.monotonic()
    told_captcha = False
    icon_clicks = 0
    last_click = -1e9
    last_note = 0
    while (elapsed := time.monotonic() - start) < FIRST_COMMENTS_WAIT_S:
        if col.main_events > 0:
            return "ok"

        if elapsed > 6 and await has_captcha(page):
            if headless:
                return "captcha_hidden"
            if not told_captcha:
                told_captcha = True
                print("  [!] Captcha terdeteksi. Selesaikan di jendela browser; skrip menunggu...")

        # Tampilan baru TikTok menyembunyikan panel komentar: klik ikon komentar.
        # Jarak minimal 12 dtk dan maksimal 5 kali supaya panel tidak terbuka-tutup terus.
        if elapsed > 8 and icon_clicks < 5 and time.monotonic() - last_click >= 12:
            if await open_comments_panel(page):
                icon_clicks += 1
                last_click = time.monotonic()
                print("  ...ikon komentar diklik untuk membuka panel komentar.")

        if elapsed - last_note >= 30:
            last_note = elapsed
            print(f"  ...menunggu komentar muncul ({int(elapsed)}/{FIRST_COMMENTS_WAIT_S} dtk). "
                  "Kalau ada pop-up login/captcha, tangani di jendela browser.")
        await asyncio.sleep(1)
    return "ok" if col.main_events > 0 else "timeout"


async def load_main_comments(page, col: Collector, max_comments: int):
    """Scroll panel komentar sampai cukup, habis, atau tidak bertambah lagi."""
    stagnant = 0
    for _ in range(MAX_SCROLL_ROUNDS):
        if len(col.main) >= max_comments:
            break
        before = len(col.main)
        try:
            await page.evaluate(JS_SCROLL_COMMENTS, SEL_COMMENT_ITEM)
            await page.mouse.wheel(0, 1500)
        except Exception:
            pass
        await asyncio.sleep(random.uniform(1.5, 3.0))

        stagnant = stagnant + 1 if len(col.main) == before else 0
        if not col.has_more and stagnant >= 1:
            break
        if stagnant >= 4:
            break
        if len(col.main) != before:
            print(f"  ...{min(len(col.main), max_comments)}/{max_comments} komentar utama")


async def expand_replies(page, col: Collector):
    """Klik tombol 'Lihat N balasan' supaya halaman sendiri yang meminta data balasan."""
    idle = 0
    for _ in range(MAX_REPLY_ROUNDS):
        before = col.reply_count()
        try:
            n = await page.evaluate(JS_MARK_REPLY_BUTTONS)
        except Exception:
            break
        if not n:
            break

        clicked = 0
        for i in range(min(n, REPLY_CLICKS_PER_ROUND)):
            btn = page.locator(f'[data-kk-reply="{i}"]').first
            try:
                await btn.scroll_into_view_if_needed(timeout=3000)
                await btn.click(timeout=3000)
                clicked += 1
            except Exception:
                continue
            await asyncio.sleep(random.uniform(1.2, 2.5))  # jeda acak supaya tidak kena rate limit

        await asyncio.sleep(random.uniform(1.5, 3.0))
        gained = col.reply_count() - before
        if gained:
            print(f"  ...total balasan terkumpul: {col.reply_count()}")
        idle = idle + 1 if (gained == 0) else 0
        if clicked == 0 or idle >= 3:
            break


# --------------------------------------------------------------------------- #
# Simpan
# --------------------------------------------------------------------------- #
def save(data: list, video_stats=None):
    df = pd.DataFrame(data)
    vdf = pd.DataFrame([video_stats]) if video_stats else pd.DataFrame()

    if not vdf.empty:
        vdf.to_csv(OUTPUT_VIDEO_CSV, index=False, encoding="utf-8-sig")
        print(f"Statistik video disimpan ke {OUTPUT_VIDEO_CSV}.")

    if df.empty:
        print("Tidak ada data komentar untuk disimpan.")
        if not vdf.empty:
            vdf.to_excel(OUTPUT_XLSX, sheet_name="statistik_video", index=False)
        return df

    try:
        df["time"] = pd.to_datetime(df["time"]).dt.tz_localize(None)
    except (TypeError, AttributeError):
        pass

    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")  # utf-8-sig: emoji/aksara aman di Excel
    with pd.ExcelWriter(OUTPUT_XLSX) as writer:
        df.to_excel(writer, sheet_name="komentar", index=False)
        if not vdf.empty:
            vdf.to_excel(writer, sheet_name="statistik_video", index=False)
    print(f"Berhasil menyimpan {len(df)} baris ke {OUTPUT_CSV} dan {OUTPUT_XLSX}.")
    return df


# --------------------------------------------------------------------------- #
# Penjalan coroutine (aman untuk terminal maupun Jupyter/VS Code notebook)
# --------------------------------------------------------------------------- #
def _run_in_new_loop(coro):
    # Windows: Playwright butuh Proactor loop (Jupyter sering memaksa Selector loop)
    loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        return loop.run_until_complete(coro)
    finally:
        try:
            loop.run_until_complete(loop.shutdown_asyncgens())
        finally:
            loop.close()


def run_async(coro):
    """asyncio.run() biasa; kalau sudah ada event loop aktif (notebook), jalankan di thread terpisah."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(_run_in_new_loop, coro).result()


# --------------------------------------------------------------------------- #
# Proses utama
# --------------------------------------------------------------------------- #
async def _run_session(ref: dict, max_comments: int, max_replies: int, with_replies: bool,
                       headless: bool, browser: str):
    """Satu sesi browser. Kembalian: (collector, statistik_video, perlu_jendela_terlihat)."""
    col = Collector(video_id=ref["id"])
    video_stats = {}
    need_visible = False

    print(f"Membuka Chrome ({'tersembunyi' if headless else 'jendela terlihat'}; "
          "pertama kali bisa agak lama)...")
    async with async_playwright() as p:
        context = await launch_context(p, browser, headless)
        try:
            page = context.pages[0] if context.pages else await context.new_page()
            col.attach(page)  # pasang perekam SEBELUM membuka halaman

            print("Membuka video...")
            try:
                await page.goto(ref["url"], wait_until="domcontentloaded", timeout=PAGE_TIMEOUT_MS)
            except Exception as e:
                detail = (str(e).strip().splitlines() or [""])[0]
                print(f"  [!] Halaman lambat/gagal dimuat: {type(e).__name__}: {detail}")
                print("      Lanjut menunggu; cek koneksi/VPN kalau tetap kosong.")

            await asyncio.sleep(random.uniform(2, 4))
            await check_landing(page, col, ref)  # juga mengisi ID untuk link pendek (vm.tiktok.com)

            # Statistik video dari data yang sudah tertanam di halaman (tanpa request tambahan)
            try:
                raw = await page.evaluate(
                    "document.getElementById('__UNIVERSAL_DATA_FOR_REHYDRATION__')?.textContent || ''"
                )
            except Exception:
                raw = ""
            video_stats = video_stats_from_item(item_from_rehydration(raw))

            try:
                print("Menunggu komentar dimuat...")
                status = await wait_for_first_comments(page, col, headless)
                if status == "timeout":
                    print("  [!] Belum ada komentar. Mencoba muat ulang halaman sekali...")
                    await page.reload(wait_until="domcontentloaded", timeout=PAGE_TIMEOUT_MS)
                    status = await wait_for_first_comments(page, col, headless)

                if status == "captcha_hidden":
                    need_visible = True
                elif status == "ok":
                    print(f"Mengambil maksimal {max_comments} komentar utama...")
                    await load_main_comments(page, col, max_comments)
                    print(f"Dapat {min(len(col.main), max_comments)} komentar utama.")
                    if with_replies and col.main:
                        print("Menelusuri balasan...")
                        await expand_replies(page, col)
                else:
                    await print_dom_hints(page)
                    try:
                        await page.screenshot(path=DEBUG_SCREENSHOT)
                        print(f"  Tangkapan layar disimpan ke {DEBUG_SCREENSHOT} untuk dicek.")
                    except Exception:
                        pass

            except (KeyboardInterrupt, asyncio.CancelledError):
                print("\nDihentikan manual. Menyimpan data yang sudah terkumpul...")

            # cadangan statistik dari /api/item/detail/ kalau data tertanam tidak ada
            if not video_stats:
                video_stats = video_stats_from_item(col.item)
        finally:
            try:
                await context.close()
            except Exception:
                pass

    return col, video_stats, need_visible


async def get_tiktok_deep_comments(
    video_ref: str,
    max_comments: int = 100,
    max_replies: int = 0,
    with_replies: bool = True,
    headless: bool = None,
    browser: str = None,
):
    if headless is None:
        headless = HEADLESS
    browser = (browser or os.environ.get("TIKTOK_BROWSER", "chrome")).lower()
    ref = parse_video_ref(video_ref)

    col, video_stats, need_visible = await _run_session(
        ref, max_comments, max_replies, with_replies, headless, browser
    )
    if need_visible:
        # captcha tidak bisa diselesaikan saat tersembunyi -> buka ulang dengan jendela terlihat
        print("\nCaptcha muncul saat browser tersembunyi. Membuka ulang dengan JENDELA TERLIHAT: "
              "selesaikan captcha di jendela itu.\n")
        col, video_stats, _ = await _run_session(
            ref, max_comments, max_replies, with_replies, False, browser
        )

    if video_stats:
        print(
            f"Statistik: views={video_stats.get('views')}, likes={video_stats.get('likes')}, "
            f"komentar={video_stats.get('comments')}, shares={video_stats.get('shares')}, "
            f"saves={video_stats.get('saves')}"
        )
    else:
        print("  [!] Statistik video tidak ditemukan.")

    data, n_main, n_replies = build_rows(col, max_comments, max_replies)

    if n_main == 0:
        print(
            "Tidak ada komentar yang terambil. Kemungkinan: captcha belum selesai, "
            "perlu login, komentar dimatikan, atau URL salah."
        )
        if col.bad_responses:
            print(f"  {col.bad_responses} respons kosong dari TikTok (tanda permintaan diblokir): "
                  "coba jaringan lain (hotspot HP) atau login di jendela browser.")
        elif col.seen_urls:
            print(f"  Endpoint yang terlihat: {sorted(set(col.seen_urls))}")
        else:
            print("  Tidak ada request komentar sama sekali: panel komentar mungkin tidak terbuka. "
                  f"Lihat {DEBUG_SCREENSHOT} dan cek SEL_COMMENT_ITEM / SEL_COMMENT_OPENERS.")

    print(f"Selesai: {n_main} komentar utama + {n_replies} balasan.")
    return save(data, video_stats)


if __name__ == "__main__":
    video_ref = input("Masukkan URL atau ID video TikTok: ").strip()
    if not video_ref:
        raise SystemExit("URL/ID video kosong.")
    max_comments = int(input("Batas komentar utama (default 100): ").strip() or 100)
    with_replies = input("Ambil balasan juga? (Y/n, default Y): ").strip().lower() not in ("n", "no", "t", "tidak")
    max_replies = 0
    if with_replies:
        max_replies = int(
            input("Batas balasan per komentar (0 = semua, default 0): ").strip() or 0
        )
    browser = (
        input("Browser (chrome/msedge/chromium, Enter = chrome): ").strip().lower()
        or os.environ.get("TIKTOK_BROWSER", "chrome")
    )

    # Jendela browser SELALU terlihat kecuali TIKTOK_HEADLESS=1 diset (tidak ada pertanyaan lagi,
    # supaya tidak salah jawab). Kalau captcha muncul saat tersembunyi, skrip membuka ulang otomatis.
    df = run_async(
        get_tiktok_deep_comments(
            video_ref, max_comments, max_replies, with_replies, HEADLESS, browser
        )
    )
    if not df.empty:
        print(df.head())

Membuka Chrome (jendela terlihat; pertama kali bisa agak lama)...
Browser: chrome
Membuka video...
  Halaman: https://www.tiktok.com/@kopikenangan.id/video/7687868581870275861
  Judul  : (15)Siap healing bareng Purple Cloud Series? 💜 Langsung gas ke outlet sek... | Kopi Kenangan | TikTok
Menunggu komentar dimuat...
Mengambil maksimal 100 komentar utama...
  ...33/100 komentar utama
  ...48/100 komentar utama
  ...74/100 komentar utama
  ...100/100 komentar utama
Dapat 100 komentar utama.
Menelusuri balasan...
  ...total balasan terkumpul: 17
  ...total balasan terkumpul: 24
Statistik: views=498100, likes=11800, komentar=373, shares=1014, saves=588
Selesai: 100 komentar utama + 24 balasan.
Statistik video disimpan ke Kopi_kenangan_tiktok_video.csv.
Berhasil menyimpan 124 baris ke Kopi_kenangan_tiktok_comentar.csv dan Kopi_kenangan_tiktok_comentar.xlsx.
            comment_id            parent_id                 type  \
0  7687931266339832596                  NaN         Main Comment   


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 323 entries, 0 to 322
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   comment_id  323 non-null    str  
 1   parent_id   223 non-null    str  
 2   type        323 non-null    str  
 3   username    323 non-null    str  
 4   comment     323 non-null    str  
 5   like_count  323 non-null    int64
 6   time        323 non-null    str  
dtypes: int64(1), str(6)
memory usage: 17.8 KB


In [ ]:
import pandas as pd
import re
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("comentar_instagram.csv", sep=";")
df.head(10)

,comment_id,parent_id,type,username,comment,like_count,time
0,"1,79043E+16",NaN,Main Comment,gwiyomi_chan,Love it 💕💕💕,8,21/09/2026 06:19
1,"1,81145E+16",NaN,Main Comment,anggunwibowo90,Cuma jangan hilangkan menu americano blueberry...,3,29/09/2026 01:23
2,"1,78695E+16","1,81145E+16",Reply to @anggunwibowo90,kopikenangan.id,"Hi Kak @anggunwibowo90, terima kasih atas masu...",0,29/09/2026 01:25
3,"1,83737E+16","1,81145E+16",Reply to @anggunwibowo90,anggunwibowo90,@kopikenangan.id iya ka soalnya udah ga keluar...,0,29/09/2026 02:12
4,"1,86226E+16",NaN,Main Comment,saraaxynn,Sebagai pencinta taro dan haechan ini adalah c...,10,22/09/2026 00:13
5,"1,81064E+16",NaN,Main Comment,citaanjani07,"@srhyniii siappp si ini mahh , mana varian ras...",3,21/09/2026 06:01
6,"1,86239E+16","1,81064E+16",Reply to @citaanjani07,srhyniii,"@citaanjani07 bingung, aku ga begitu suka taro😭",0,21/09/2026 06:02
7,"1,79584E+16","1,81064E+16",Reply to @citaanjani07,citaanjani07,"@srhyniii ehhh iya aku kira suka, ini lebih am...",0,21/09/2026 06:13
8,"1,81384E+16",NaN,Main Comment,wendakio,Siapa kmrn yg kebakaran perkara lope ungu wkw,1386,21/09/2026 06:17
9,"1,80959E+16","1,81384E+16",Reply to @wendakio,rabbitdoyna,@wendakio si paling ungu 😂,44,21/09/2026 06:22


In [ ]:
df = df[['username', 'comment', 'time']]
df

,username,comment,time
0,gwiyomi_chan,Love it 💕💕💕,21/09/2026 06:19
1,anggunwibowo90,Cuma jangan hilangkan menu americano blueberry...,29/09/2026 01:23
2,kopikenangan.id,"Hi Kak @anggunwibowo90, terima kasih atas masu...",29/09/2026 01:25
3,anggunwibowo90,@kopikenangan.id iya ka soalnya udah ga keluar...,29/09/2026 02:12
4,saraaxynn,Sebagai pencinta taro dan haechan ini adalah c...,22/09/2026 00:13
...,...,...,...
318,_nsa.airunikha,@aaalleekaa 👉👈,25/09/2026 18:25
319,aaalleekaa,@_nsa.airunikha maksud,26/09/2026 01:02
320,mordemoore,@kumaimoru,24/09/2026 10:58
321,kumaimoru,@mordemoore mau.. 🫩🫩,24/09/2026 22:41


In [ ]:
# Case folding dan Hapus duplikat
df['comment'] = df['comment'].str.lower()
df = df.drop_duplicates(subset=['comment'])

# Membersihkan teks (Cleansing)
def clean_comment(text):
    text = re.sub(r'@[A-Za-z0-9_]+', '', str(text))
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'[^A-Za-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['comment_clean'] = df['comment'].apply(clean_comment)

# Normalisasi Kata
try:
    normalized_word = pd.read_excel("kamus perbaikan kata.xlsx")
    normalized_word_dict = {}
    for index, row in normalized_word.iterrows():
        if row.iloc[0] not in normalized_word_dict:
            normalized_word_dict[row.iloc[0]] = row.iloc[1]

    def normalized_term(document):
        return [normalized_word_dict[term] if term in normalized_word_dict else term for term in str(document).split()]
        
    df['comment_norm'] = df['comment_clean'].apply(normalized_term)
except FileNotFoundError:
    print("File kamus perbaikan kata.xlsx tidak ditemukan, melewati tahap normalisasi.")
    df['comment_norm'] = df['comment_clean'].apply(lambda x: str(x).split())

df[['comment', 'comment_clean', 'comment_norm']].head()

,comment,comment_clean,comment_norm
0,love it 💕💕💕,love it,"[love, it]"
1,cuma jangan hilangkan menu americano blueberry...,cuma jangan hilangkan menu americano blueberry...,"[cuma, jangan, hilangkan, menu, americano, blu..."
2,"hi kak @anggunwibowo90, terima kasih atas masu...",hi kak terima kasih atas masukkan yang diberik...,"[hi, kak, terima, kasih, atas, masukkan, yang,..."
3,@kopikenangan.id iya ka soalnya udah ga keluar...,id iya ka soalnya udah ga keluar lagi menu yan...,"[id, iya, ka, soalnya, sudah, tidak, keluar, l..."
4,sebagai pencinta taro dan haechan ini adalah c...,sebagai pencinta taro dan haechan ini adalah c...,"[sebagai, pencinta, taro, dan, haechan, ini, a..."


In [ ]:
import matplotlib.pyplot as plt
import nltk
from textblob import TextBlob
from wordcloud import WordCloud, STOPWORDS
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory, StopWordRemover, ArrayDictionary
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

In [ ]:
# Stopword Removal
more_stop_words = ["tidak"]
stop_words = StopWordRemoverFactory().get_stop_words()
stop_words.extend(more_stop_words)
new_array = ArrayDictionary(stop_words)
stop_words_remover_new = StopWordRemover(new_array)

def stopword(str_text):
    str_text = " ".join(str_text)
    str_text = stop_words_remover_new.remove(str_text)
    return str_text

df['comment_norm'] = df['comment_norm'].apply(lambda x: stopword(x))

# Stemming
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stemming(text_cleaning):
    do = []
    for w in text_cleaning.split():
        dt = stemmer.stem(w)
        do.append(dt)
    d_clean = " ".join(do)
    return d_clean

df['comment_norm'] = df['comment_norm'].apply(stemming)
df.dropna(subset=['comment_norm'], inplace=True)
df['comment_norm'] = df['comment_norm'].astype(str)

df[['comment_norm']].head()